## Ollama (Modelos locales)

### Instalar Ollama

Ese código prepara un entorno Linux/Colab para instalar y verificar Ollama:  
Primero actualiza los paquetes del sistema e instala zstd, una herramienta de compresión/descompresión rápida usada habitualmente para archivos .zst;  
Luego descarga y ejecuta el script oficial de instalación de Ollama mediante curl, siguiendo el método recomendado para Linux;   
Después evita instalar cuda-drivers porque no es indispensable si Ollama se va a usar por CPU o si la instalación de drivers GPU daba errores;  
Verifica que Ollama quedó instalado ejecutando /usr/local/bin/ollama --version;  

Finalmente define la variable LD_LIBRARY_PATH apuntando a librerías NVIDIA, lo cual puede servir si más adelante se intenta usar GPU, aunque no resuelve por sí solo problemas como “ollama command not found”

In [ ]:
# Install zstd, a required dependency for Ollama extraction
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

# The 'cuda-drivers' installation previously failed and is not essential for basic Ollama functionality (CPU).
# Removing related lines to avoid errors and simplify the installation process.
# !echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
# !sudo apt-get update && sudo apt-get install -y cuda-drivers

# Verify Ollama installation. Using full path for robustness.
# This confirms the binary is available at the expected location.
!/usr/local/bin/ollama --version

# This environment variable setting is typically for NVIDIA GPU libraries.
# Keeping it if the user intends GPU usage later, but it's not the root cause of 'ollama command not found'.
import os
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,406 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

### Iniciar servidor

Ese comando inicia el servidor de Ollama en segundo plano:  
ollama serve levanta el servicio local que permite ejecutar modelos y recibir pedidos desde la API local, normalmente en http://localhost:11434; /usr/local/bin/ollama usa la ruta completa del ejecutable para evitar problemas de PATH;  
nohup hace que el proceso siga corriendo aunque se cierre la terminal o finalice la celda;  
El & lo manda al fondo para que la notebook no quede bloqueada. En Colab, esto sirve para dejar Ollama **“escuchando”** mientras en otra celda ejecutás comandos como ollama run ... o llamadas HTTP a la API local.

In [ ]:
!nohup /usr/local/bin/ollama serve &

nohup: appending output to 'nohup.out'


Extraer el modelo Gemma3

In [ ]:
# Seleccionar modelos acá https://ollama.com/search
# IMPORTANTE: Tener en cuenta el hardware a utilizar
!/usr/local/bin/ollama pull gemma3:1b

### Instalar el paquete Ollama

Ese comando instala en el entorno de Python el cliente oficial de Ollama para Python. No instala el programa Ollama como servidor; eso ya lo hiciste antes con curl ... | sh.  
Lo que hace pip install ollama es permitir que desde una notebook o script puedas escribir import ollama y comunicarte con el servidor local de Ollama, por ejemplo para chatear con un modelo, generar texto, listar modelos o pedir embeddings. Según la documentación del paquete, Ollama debe estar instalado y corriendo previamente, y también necesitás tener descargado algún modelo, por ejemplo con ollama pull gemma3.  

Resumiendo: **!pip install ollama** instala la librería Python para controlar Ollama desde código, mientras que **ollama serve** es lo que mantiene activo el servidor local que realmente ejecuta los modelos.

In [ ]:
!pip install ollama

### Prueba de ejecución

In [1]:
import ollama

response = ollama.chat(
    model='gemma3:1b',
    messages=[
        {
            'role': 'user',
          'content': "¿Quién fue Pasteur?",
        },
    ]
)

print(response['message']['content'])

Louis Pasteur fue un científico y químico francés reconocido por sus descubrimientos revolucionarios en la microbiología, la química y la fisiología. A menudo se le considera **uno de los científicos más importantes de la historia**, y su trabajo sentó las bases para gran parte de la moderna medicina y la agricultura.

Aquí te resumo sus principales logros y contribuciones:

**1. La Teoría de los Germes (1860):**

* **El descubrimiento de que las bacterias no son necesarias para causar enfermedades y que pueden ser causadas por microorganismos específicos.** Esta fue una idea revolucionaria y crucial.
* **El concepto de la "vacuna":** Pasteur comprendió que la enfermedad se producía cuando las bacterias contaminaban un cuerpo. Así, desarrolló la idea de usar sustancias que podían "desarticar" las bacterias, permitiendo al cuerpo defenderse.
* **La primera vacuna moderna:** Pasteur desarrolló la primera vacuna contra la viruela, que le valió el Premio Nobel de Medicina en 1863.

**2. La

### Ejecución con streaming (va mostrando a medida que se genera)


In [2]:
from ollama import chat

stream = chat(
    model='gemma3:1b',
    messages=[
        {
            'role': 'user',
            'content': '¿Quién fue Pasteur?'
        }
    ],
    stream=True,
)

for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Louis Pasteur fue un científico francés considerado una figura clave en la historia de la química y la biología, y es famoso por sus descubrimientos revolucionarios sobre la microbiología. Aquí te dejo un resumen de quién fue y por qué fue tan importante:

**1. Origen y Vida Previa:**

* **Nacimiento:** Nació en 1822 en la Francia rural, en una familia de campesinos.
* **Educación:**  Aunque recibió una educación formal limitada, demostró un gran interés por la ciencia y la experimentación desde joven.
* **Trabajo en el Hospital de París:**  Después de obtener su título en medicina, Pasteur trabajó como asistente en el Hospital de París, donde se destacó por su inteligencia y capacidad para realizar experimentos.

**2.  Descubrimientos Revolucionarios:**

* **La Teoría de la Germinación:**  Pasteur, junto con Robert Koch, desarrolló la teoría de la germinación. Esto significa que las enfermedades se transmiten no por causas de infección, sino por la presencia de microorganismos (bacter

## Gemini API (Cloud models)

---



### Instalar librería

Ese comando instala o actualiza en Python la librería google-generativeai, **que permite usar la API de Gemini desde una notebook o script**.  
En Colab, el signo ! indica que se ejecuta como comando de terminal; pip install instala el paquete; -q significa quiet, es decir, muestra menos mensajes durante la instalación; y -U significa upgrade, o sea que actualiza el paquete si ya estaba instalado.

In [ ]:
#!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.2.2 requires google-genai<2.0.0,>=1.65.0, but you have google-genai 2.2.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.2.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.2.0 which is incompatible.


### Importamos la librería

In [3]:
import google.generativeai as genai

/tmp/ipykernel_12222/613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Seteamos la API Key de Google

Anteriormente debemos generar una API Key en https://aistudio.google.com/app/api-keys

In [4]:
#from google.colab import userdata

#GOOGLE_API_KEY=userdata.get('nlp')
# Obtener la clave desde archivo config.ini
import configparser
config = configparser.ConfigParser()
config.read('config.ini')
GOOGLE_API_KEY = config['GOOGLE']['API_KEY']
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
#GOOGLE_API_KEY

### Listamos los modelos disponibles

Ese código recorre la lista de modelos disponibles para tu API key de Gemini y muestra solamente aquellos que soportan el método generateContent, que es el método usado para generar respuestas a partir de texto, imágenes, audio u otros contenidos. En la documentación oficial, Google define supportedGenerationMethods como la lista de métodos de generación soportados por cada modelo, donde puede aparecer generateContent.

In [5]:
print("Lista de modelos que soportan generateContent:\n")
for m in genai.list_models():
    for action in m.supported_generation_methods:
        if action == "generateContent":
            print(m.name)


Lista de modelos que soportan generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-resear

Una vez visto los modelos podemos ver su Rate Limit en https://ai.google.dev/gemini-api/docs/rate-limits?hl=es-419#current-rate-limits

### Inicializamos el modelo

In [6]:
# Seleccionamos el modelo en
model = genai.GenerativeModel('models/gemini-2.5-flash')

| Modelo                    |     Plan |    RPM |         TPM |     RPD |
| ------------------------- | -------: | -----: | ----------: | ------: |
| `models/gemini-2.5-flash` | Gratuito | **10** | **250.000** | **250** |

10 solicitudes por minuto **RPM**  
250.000 tokens de entrada por minuto **TPM**  
250 solicitudes por día **RPD**  


Ojo con un detalle importante: aunque esa es la referencia general, Google indica que los límites reales pueden variar según el proyecto, el nivel de uso y el estado de la cuenta; recomienda revisar los límites activos en AI Studio.

En la práctica, si estás usando esto en Colab o una app chica, el límite que probablemente te corte primero es el de 250 requests por día o el de 10 requests por minuto

### Código de ejemplo

In [7]:
response = model.generate_content("Respondeme lo anterior que te pregunté en inglés")
print(response.text)

Claro, para responder a lo anterior en inglés, por favor recuérdame cuál fue la pregunta o el tema al que te refieres. Como soy una IA, no tengo memoria de conversaciones pasadas a menos que me lo proporciones en el mismo hilo.

Una vez que me digas la pregunta o el tema, con gusto te daré la respuesta en inglés.


## Integrar con Langchain


**LangChain es un framework para construir aplicaciones basadas en modelos de lenguaje, como chatbots, asistentes, sistemas de preguntas y respuestas, RAG, agentes o flujos de automatización con IA.**  
Su función principal es **permitir conectar de manera ordenada distintos componentes:** prompts, modelos, documentos, bases vectoriales, herramientas externas, memoria, parsers y pasos intermedios.  
En lugar de hacer una llamada aislada a un modelo, LangChain **permite diseñar una secuencia de trabajo reutilizable**, por ejemplo: recibir una pregunta, buscar información relevante, construir un prompt, enviarlo al modelo y devolver una respuesta estructurada.


### Instalar librerías

In [ ]:
#!pip install langchain
#!pip install langchain-community
#!pip install langchain-google-genai

### Importar Langchain e instanciar modelo


In [8]:
from langchain_community.chat_models import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

use_ollama_instead_gemini = False
llm = None
if use_ollama_instead_gemini:
    llm = ChatOllama(model="gemma3:1b")
else:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY)

### Manejar instrucciones y templates


**Este código compara dos formas de construir una cadena en LangChain:**  

Primero crea un prompt simple con `ChatPromptTemplate.from_template("Cuentame sobre {topic}")`, donde `{topic}` se reemplaza por `"el sol"` y el modelo recibe una instrucción directa del usuario; luego crea otro prompt con `ChatPromptTemplate.from_messages`, separando un mensaje de sistema y un mensaje de usuario.  

En la segunda cadena, el mensaje de sistema define el comportamiento general del modelo —responder de manera sencilla y en inglés—, mientras que el mensaje de usuario solo aporta el tópico, en este caso `"la luna"`.  

**La diferencia principal es que el primer caso depende solo de la instrucción del usuario, mientras que el segundo usa una estructura conversacional más controlada, donde el sistema fija reglas generales y el usuario entrega la tarea específica**.


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence

# Step 1 & 2: Definir un template y una cadena de input de usuario
prompt_user = ChatPromptTemplate.from_template("Cuentame sobre {topic}")
chain_user = prompt_user | llm

# Step 3: Invocar esa cadena con un ejemplo
response_user = chain_user.invoke({"topic": "el sol"})
print("Respuesta solamente con el mensaje del usuario:")
print(response_user.content)

print("-"*120)

# Step 4 & 5: Definir instrucciones de sistema y usuario y crear una nueva cadena
prompt_system_user = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de mucha ayuda que responde de manera sencilla lo que el usuario le pida. El usuario solo enviará un tópico y tu debes trabajarlo. Responde en inglés"),
    ("user", "{topic}")
])
chain_system_user = prompt_system_user | llm

# Step 6: Invocar la nueva cadena con ejemplos
response_system_user = chain_system_user.invoke({"topic": "la luna"})
print("\nRespuesta con el mensaje del sistema y del usuario:")
print(response_system_user.content)

Respuesta solamente con el mensaje del usuario:
¡Claro que sí! El Sol es la estrella más importante para nosotros, el corazón de nuestro sistema solar. Aquí te cuento sobre él:

**1. ¿Qué es el Sol?**
Es la **estrella central de nuestro sistema solar**, una esfera gigantesca de plasma caliente y brillante. Es la fuente fundamental de energía para la vida en la Tierra, proporcionando luz, calor y regulando nuestro clima.

**2. Tipo de Estrella:**
Se clasifica como una **enana amarilla** (o más precisamente, una estrella de secuencia principal de tipo G). Aunque se le llama "amarilla", su luz es en realidad blanca; la atmósfera de la Tierra la dispersa haciendo que la veamos amarillenta, especialmente al amanecer o atardecer.

**3. Tamaño y Masa:**
*   **Diámetro:** Es aproximadamente **109 veces el diámetro de la Tierra**.
*   **Volumen:** ¡Podrían caber más de **1.3 millones de Tierras** dentro del Sol!
*   **Masa:** Constituye el **99.8% de la masa total de todo el sistema solar**. Su

### LLM Como clasificador

Este código construye una cadena en LangChain para realizar una **clasificación de sentimiento** sobre una reseña. Primero se define un `ChatPromptTemplate` con dos mensajes: uno de sistema, que le indica al modelo que debe clasificar la reseña en una de tres categorías posibles —`Mala`, `Buena` o `Neutra`— y devolver únicamente la categoría; y uno de usuario, que contiene la variable `{mensaje}`, donde se insertará el texto a analizar. Luego, mediante `prompt_user | llm`, se crea la cadena que conecta el prompt con el modelo. Al invocar la cadena con el mensaje `"No puedo dejar de disfrutar el producto"`, el modelo debe interpretar el sentido positivo de la frase y responder solamente con la categoría correspondiente, probablemente.

In [10]:
prompt_user = ChatPromptTemplate.from_messages([
    ("system", "Clasifica la siguiente reseña entre 3 categorías distintas las cuales son 'Mala', 'Buena', 'Neutra'. Devuelve solo la categoria"),
    ("user", "{mensaje}")
])
chain_user = prompt_user | llm

# Step 3: Invocar esa cadaena con un ejemplo
response_user = chain_user.invoke({"mensaje": "No puedo dejar de disfrutar el producto"})
print("Respuesta de la clasificación:")
print(response_user.content)

Respuesta de la clasificación:
Buena


### Trabajando con información sobre un documento

In [11]:
docs = """Lionel Andrés Messi Cuccittini (Rosario, Santa Fe, Argentina 24 de junio de 1987), conocido como Leo Messi, es un futbolista argentino que juega como delantero o centrocampista. Desde 2023, integra el plantel del Inter Miami de la MLS canadoestadounidense. Es también internacional con la selección de Argentina, de la que es capitán.

Con el Fútbol Club Barcelona, al que estuvo ligado más de veinte años, ganó 35 títulos, entre ellos,diez de La Liga, cuatro de la Liga de Campeones de la UEFA y siete de la Copa del Rey.

Considerado con frecuencia el mejor jugador del mundo y uno de los mejores de todos los tiempos,[10]​ es el único en la historia que ha ganado, entre otras distinciones, ocho veces el Balón de Oro, ocho premios de la FIFA al mejor jugador del mundo, seis Botas de Oro y dos Balones de Oro de la Copa Mundial de Fútbol. En 2020, se convirtió en el primer futbolista y el primer argentino en recibir un premio Laureus y fue incluido en el Dream Team del Balón de Oro.

Tiene, entre otros, los récords por más goles en una temporada,[11]​ en un mismo club y en un año calendario. Es, además, el máximo goleador histórico del Barcelona y de la selección argentina, de La Liga, la Supercopa de España, la Supercopa de Europa y el jugador no europeo con más goles en la Liga de Campeones de la UEFA.

Nacido y criado en la ciudad de Rosario, a los 13 años se radicó en España, donde el Barcelona accedió a pagar el tratamiento de la enfermedad hormonal que le habían diagnosticado de niño. Después de una rápida progresión por la Academia juvenil del Barcelona, hizo su debut oficial con el primer equipo en octubre de 2004, a los diecisiete años. A pesar de haber sido propenso a lesiones en los inicios de su carrera, ya en 2006 se estableció como jugador fundamental para el club. Su primera temporada ininterrumpida fue la 2008-09, en la que el Barcelona alcanzó el primer triplete del fútbol español. Por su estilo de juego de pequeño driblador zurdo,[12]​ pronto se lo comparó con su compatriota Diego Maradona quien, en 2007, lo declaró su «sucesor».

En 2009, a los veintidós años, ganó su primer Balón de Oro y el premio al Jugador Mundial de la FIFA del año. Siguieron tres temporadas exitosas, en las que ganó cuatro Balones de Oro de forma consecutiva, hecho que no tenía precedentes. Hasta el momento, su mejor campaña personal fue en 2011-12, cuando estableció el récord de más goles en una temporada, tanto en La Liga como en otras competiciones europeas. Durante las dos siguientes temporadas, también sufrió lesiones y, en 2014, perdió el Balón de Oro frente a Cristiano Ronaldo, a quien se considera su rival. Recuperó su mejor forma durante la campaña 2014-15, en la que superó los registros de máximo goleador absoluto en La Liga y la Liga de Campeones y logró con el Barcelona un histórico segundo triplete, además de ganar su quinto Balón de Oro. Volvería a ganarlo en 2019, 2021 y 2023.

Como internacional argentino, ha representado a su país en catorce torneos mayores. A nivel juvenil, en 2005 participó con la selección sub-20 en el Sudamericano de Colombia y ganó la Copa Mundial de Países Bajos, torneo en el que finalizó como mejor jugador y máximo goleador y, con la sub-23, recibió la medalla de oro en los Juegos Olímpicos de 2008. Después de debutar en la selección mayor en agosto de 2005, en el Mundial de Alemania 2006 se convirtió en el argentino más joven en jugar y en marcar en un mundial. Al año siguiente, en la Copa América, fue nombrado mejor jugador joven del torneo. Como capitán desde agosto de 2011, llegó con su equipo a las finales del Mundial de Brasil 2014, de la Copa América 2015 y de la Copa América Centenario, además de ganar la Copa América 2021 ante Brasil en el Maracaná, la Finalissima 2022 frente a Italia en Wembley, el Mundial de Catar 2022 contra Francia en el estadio Lusail y la Copa América 2024 ante Colombia en el Hard Rock Stadium."""

len(docs)

3921

#### Ejemplo 1: Pregunta directa con contexto incluido en el prompt

Este código construye una cadena en LangChain para responder preguntas usando un **documento como contexto**.   

Primero se crea un `ChatPromptTemplate` con un mensaje de sistema que le indica al modelo que debe responder únicamente basándose en el contenido de `{document}` y que, si la información no aparece allí, debe decir que no la tiene.   

Luego se agrega el mensaje de usuario mediante `{question}`, que representa la pregunta concreta. Después, con `prompt_with_context | llm`, se crea la cadena que une el prompt con el modelo.  

Finalmente, la cadena se invoca tres veces pasando el mismo documento `docs` y distintas preguntas sobre Messi, como dónde y cuándo nació, cuántos Balones de Oro ganó y qué torneos obtuvo con la selección argentina. La idea central es mostrar cómo el modelo puede responder condicionado por un texto de referencia, evitando depender solo de su conocimiento interno.


In [12]:
# Creamos un template que incluya el documento y la pregunta del usuario
prompt_with_context = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente experto que responde preguntas basándose únicamente en el siguiente documento:\n\n{document}\n\nSi la información no está en el documento, indica que no tienes esa información."),
    ("user", "{question}")
])

# Creamos la cadena
chain_with_context = prompt_with_context | llm

# Ejemplo 1: Pregunta sobre nacimiento
response1 = chain_with_context.invoke({
    "document": docs,
    "question": "¿Dónde y cuándo nació Messi?"
})
print("Pregunta 1: ¿Dónde y cuándo nació Messi?")
print(f"Respuesta: {response1.content}\n")

# Ejemplo 2: Pregunta sobre títulos
response2 = chain_with_context.invoke({
    "document": docs,
    "question": "¿Cuántos Balones de Oro ganó Messi?"
})
print("Pregunta 2: ¿Cuántos Balones de Oro ganó Messi?")
print(f"Respuesta: {response2.content}\n")

# Ejemplo 3: Pregunta sobre selección
response3 = chain_with_context.invoke({
    "document": docs,
    "question": "¿Qué torneos ganó con la selección argentina?"
})
print("Pregunta 3: ¿Qué torneos ganó con la selección argentina?")
print(f"Respuesta: {response3.content}")

Pregunta 1: ¿Dónde y cuándo nació Messi?
Respuesta: Lionel Andrés Messi Cuccittini nació en Rosario, Santa Fe, Argentina, el 24 de junio de 1987.

Pregunta 2: ¿Cuántos Balones de Oro ganó Messi?
Respuesta: Messi ha ganado ocho veces el Balón de Oro.

Pregunta 3: ¿Qué torneos ganó con la selección argentina?
Respuesta: Con la selección argentina, Lionel Messi ganó los siguientes torneos:

*   **Copa Mundial de Países Bajos** (con la selección sub-20 en 2005)
*   **Medalla de oro en los Juegos Olímpicos de 2008** (con la selección sub-23)
*   **Copa América 2021**
*   **Finalissima 2022**
*   **Mundial de Catar 2022**
*   **Copa América 2024**


#### Ejemplo 2: Resumen del documento

In [13]:
# Generamos un resumen del documento
prompt_summary = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que genera resúmenes concisos y claros."),
    ("user", "Resume el siguiente texto en 3-4 oraciones, destacando los puntos más importantes:\n\n{document}")
])

chain_summary = prompt_summary | llm
response_summary = chain_summary.invoke({"document": docs})

print("=== RESUMEN DEL DOCUMENTO ===\n")
print(response_summary.content)

=== RESUMEN DEL DOCUMENTO ===

Lionel Messi es un futbolista argentino, capitán de su selección y actual jugador del Inter Miami. Es reconocido como uno de los mejores de todos los tiempos, habiendo ganado un récord de ocho Balones de Oro y ocho premios FIFA al mejor jugador. Con el FC Barcelona, club al que estuvo ligado por más de veinte años, conquistó 35 títulos, incluyendo diez Ligas y cuatro Ligas de Campeones. Además, ha liderado a Argentina a importantes victorias internacionales, como la Copa América 2021, la Finalissima 2022, el Mundial de Catar 2022 y la Copa América 2024.


#### Ejemplo 3: Extracción de datos estructurados

In [14]:
# Extraemos información estructurada del documento
prompt_extraction = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que extrae información específica de documentos."),
    ("user", """Del siguiente documento, extrae la siguiente información en formato de JSON:

- Nombre completo -> name
- Fecha de nacimiento -> birth_date
- Lugar de nacimiento -> birth_place
- Club actual (a partir de 2023) -> club
- Número de Balones de Oro ganados -> ballon_d_or
- Principales torneos ganados con Argentina -> torneos

Documento:
{document}""")
])

chain_extraction = prompt_extraction | llm
response_extraction = chain_extraction.invoke({"document": docs})

print("=== INFORMACIÓN EXTRAÍDA ===\n")
print(response_extraction.content)

=== INFORMACIÓN EXTRAÍDA ===

```json
{
  "name": "Lionel Andrés Messi Cuccittini",
  "birth_date": "24 de junio de 1987",
  "birth_place": "Rosario, Santa Fe, Argentina",
  "club": "Inter Miami",
  "ballon_d_or": 8,
  "torneos": [
    "Copa Mundial de Países Bajos (Sub-20) 2005",
    "Juegos Olímpicos de 2008 (Medalla de oro con la Sub-23)",
    "Copa América 2021",
    "Finalissima 2022",
    "Mundial de Catar 2022",
    "Copa América 2024"
  ]
}
```


#### Ejemplo 4: Verificación de información

In [15]:
# Verificamos afirmaciones contra el documento
prompt_factcheck = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de verificación de hechos.
    Dado el siguiente documento, verifica si las afirmaciones son VERDADERAS, FALSAS o NO SE PUEDE VERIFICAR.

    Documento:
    {document}"""),
    ("user", "Verifica esta afirmación: {statement}")
])

chain_factcheck = prompt_factcheck | llm

# Afirmaciones para verificar
afirmaciones = [
    "Messi ganó 8 Balones de Oro",
    "Messi nació en Buenos Aires",
    "Messi ganó el Mundial 2022 con Argentina",
    "Messi tiene 5 hijos",
    "Messi jugó más de 20 años en el Barcelona y convirtió 500.000 goles"
]

print("=== VERIFICACIÓN DE AFIRMACIONES ===\n")
for afirmacion in afirmaciones:
    response = chain_factcheck.invoke({
        "document": docs,
        "statement": afirmacion
    })
    print(f"Afirmación: {afirmacion}")
    print(f"Verificación: {response.content}\n")
    print("-" * 80 + "\n")

=== VERIFICACIÓN DE AFIRMACIONES ===

Afirmación: Messi ganó 8 Balones de Oro
Verificación: VERDADERA

--------------------------------------------------------------------------------

Afirmación: Messi nació en Buenos Aires
Verificación: FALSA. El documento indica que Messi nació en Rosario, Santa Fe, Argentina.

--------------------------------------------------------------------------------

Afirmación: Messi ganó el Mundial 2022 con Argentina
Verificación: VERDADERA.

El documento indica: "Como capitán desde agosto de 2011, llegó con su equipo a las finales del Mundial de Brasil 2014, de la Copa América 2015 y de la Copa América Centenario, además de ganar la Copa América 2021 ante Brasil en el Maracaná, la Finalissima 2022 frente a Italia en Wembley, el Mundial de Catar 2022 contra Francia en el estadio Lusail y la Copa América 2024 ante Colombia en el Hard Rock Stadium."

--------------------------------------------------------------------------------

Afirmación: Messi tiene 5 h

#### Ejemplo 5: Sistema de chat con memoria conversacional (historial de mensajes)

In [16]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Inicializamos el historial de mensajes con el contexto del documento
# El primer mensaje es el SYSTEM que contiene el documento como contexto
chat_history = [
    SystemMessage(content=f"""Eres un asistente experto que responde preguntas basándose en el siguiente documento sobre Lionel Messi.
    Debes recordar toda la conversación anterior para responder preguntas que hagan referencia a mensajes previos.

    DOCUMENTO:
    {docs}

    Responde de manera concisa y precisa basándote únicamente en este documento.""")
]

def chat_with_memory(user_message):
    """
    Función que mantiene el historial de conversación y permite preguntas contextuales

    Args:
        user_message: El mensaje del usuario

    Returns:
        La respuesta del modelo basada en el documento y el historial
    """
    # Agregamos el mensaje del usuario al historial
    chat_history.append(HumanMessage(content=user_message))

    # Invocamos el modelo con TODO el historial de mensajes
    response = llm.invoke(chat_history)

    # Agregamos la respuesta del asistente al historial
    chat_history.append(AIMessage(content=response.content))

    return response.content

# CONVERSACIÓN CON MEMORIA
print("=== CHAT CON MEMORIA CONVERSACIONAL ===\n")
print("Nota: El modelo recuerda todo el historial de la conversación\n")
print("=" * 80 + "\n")

# Pregunta 1: Información básica
print("👤 Usuario: ¿Cuál es el nombre completo del jugador del que me hablas?")
respuesta1 = chat_with_memory("¿Cuál es el nombre completo del jugador del que me hablas?")
print(f"🤖 Asistente: {respuesta1}\n")
print("-" * 80 + "\n")

# Pregunta 2: Hace referencia a la pregunta anterior ("él")
print("👤 Usuario: ¿Dónde nació él?")
respuesta2 = chat_with_memory("¿Dónde nació él?")
print(f"🤖 Asistente: {respuesta2}\n")
print("-" * 80 + "\n")

# Pregunta 3: Pide traducción de información anterior
print("👤 Usuario: ¿Me puedes decir eso en inglés?")
respuesta3 = chat_with_memory("¿Me puedes decir eso en inglés?")
print(f"🤖 Asistente: {respuesta3}\n")
print("-" * 80 + "\n")

# Pregunta 4: Nueva información específica
print("👤 Usuario: ¿Cuántos Balones de Oro ganó?")
respuesta4 = chat_with_memory("¿Cuántos Balones de Oro ganó?")
print(f"🤖 Asistente: {respuesta4}\n")
print("-" * 80 + "\n")

# Pregunta 5: Hace referencia a la respuesta anterior ("esos premios")
print("👤 Usuario: ¿En qué años ganó esos premios?")
respuesta5 = chat_with_memory("¿En qué años ganó esos premios?")
print(f"🤖 Asistente: {respuesta5}\n")
print("-" * 80 + "\n")

# Pregunta 6: Pregunta sobre información previa diferente
print("👤 Usuario: Volviendo a su lugar de nacimiento, ¿en qué país está esa ciudad?")
respuesta6 = chat_with_memory("Volviendo a su lugar de nacimiento, ¿en qué país está esa ciudad?")
print(f"🤖 Asistente: {respuesta6}\n")
print("-" * 80 + "\n")

# Pregunta 7: Resumen completo de la conversación en otro idioma
print("👤 Usuario: Me puedes traducir todo lo que vimos en portugues?")
respuesta6 = chat_with_memory("Me puedes traducir todo lo que vimos en portugues?")
print(f"🤖 Asistente: {respuesta6}\n")
print("-" * 80 + "\n")

# Mostramos cuántos mensajes hay en el historial
print(f"\n📊 ESTADÍSTICAS DEL CHAT:")
print(f"Total de mensajes en el historial: {len(chat_history)}")
print(f"  - 1 mensaje del sistema (contexto del documento)")
print(f"  - {(len(chat_history) - 1) // 2} mensajes del usuario")
print(f"  - {(len(chat_history) - 1) // 2} respuestas del asistente")
print(f"\n💡 El modelo tiene acceso a TODOS estos mensajes en cada nueva pregunta.")

=== CHAT CON MEMORIA CONVERSACIONAL ===

Nota: El modelo recuerda todo el historial de la conversación


👤 Usuario: ¿Cuál es el nombre completo del jugador del que me hablas?
🤖 Asistente: El nombre completo del jugador es Lionel Andrés Messi Cuccittini.

--------------------------------------------------------------------------------

👤 Usuario: ¿Dónde nació él?
🤖 Asistente: Nació en Rosario, Santa Fe, Argentina.

--------------------------------------------------------------------------------

👤 Usuario: ¿Me puedes decir eso en inglés?
🤖 Asistente: He was born in Rosario, Santa Fe, Argentina.

--------------------------------------------------------------------------------

👤 Usuario: ¿Cuántos Balones de Oro ganó?
🤖 Asistente: Ganó ocho Balones de Oro.

--------------------------------------------------------------------------------

👤 Usuario: ¿En qué años ganó esos premios?
🤖 Asistente: Ganó los Balones de Oro en los años: 2009, 2010, 2011, 2012, 2015 (después de la campaña 2014-15)

In [17]:
# Visualizamos el historial completo para entender la estructura
print("=== HISTORIAL COMPLETO DE LA CONVERSACIÓN ===\n")

for i, message in enumerate(chat_history):
    if isinstance(message, SystemMessage):
        print(f"[{i}] 🔧 SYSTEM:")
        # Mostramos solo los primeros 200 caracteres del documento para no llenar la pantalla
        print(f"    {message.content[:200]}...")
    elif isinstance(message, HumanMessage):
        print(f"[{i}] 👤 USUARIO:")
        print(f"    {message.content}")
    elif isinstance(message, AIMessage):
        print(f"[{i}] 🤖 ASISTENTE:")
        print(f"    {message.content}")
    print()

=== HISTORIAL COMPLETO DE LA CONVERSACIÓN ===

[0] 🔧 SYSTEM:
    Eres un asistente experto que responde preguntas basándose en el siguiente documento sobre Lionel Messi.
    Debes recordar toda la conversación anterior para responder preguntas que hagan referencia ...

[1] 👤 USUARIO:
    ¿Cuál es el nombre completo del jugador del que me hablas?

[2] 🤖 ASISTENTE:
    El nombre completo del jugador es Lionel Andrés Messi Cuccittini.

[3] 👤 USUARIO:
    ¿Dónde nació él?

[4] 🤖 ASISTENTE:
    Nació en Rosario, Santa Fe, Argentina.

[5] 👤 USUARIO:
    ¿Me puedes decir eso en inglés?

[6] 🤖 ASISTENTE:
    He was born in Rosario, Santa Fe, Argentina.

[7] 👤 USUARIO:
    ¿Cuántos Balones de Oro ganó?

[8] 🤖 ASISTENTE:
    Ganó ocho Balones de Oro.

[9] 👤 USUARIO:
    ¿En qué años ganó esos premios?

[10] 🤖 ASISTENTE:
    Ganó los Balones de Oro en los años: 2009, 2010, 2011, 2012, 2015 (después de la campaña 2014-15), 2019, 2021 y 2023.

[11] 👤 USUARIO:
    Volviendo a su lugar de nacimiento,

### Chat interactivo

In [ ]:
# Reiniciamos el historial para un nuevo chat
chat_history_interactive = [
    SystemMessage(content=f'''Eres un asistente experto que responde preguntas basándose en el siguiente documento sobre Lionel Messi.
    Debes recordar toda la conversación anterior para responder preguntas que hagan referencia a mensajes previos.

    DOCUMENTO:
    {docs}

    Responde de manera concisa y precisa basándote únicamente en este documento.''')
]

def chat_interactive(user_message):
    chat_history_interactive.append(HumanMessage(content=user_message))
    response = llm.invoke(chat_history_interactive)
    chat_history_interactive.append(AIMessage(content=response.content))
    return response.content

print("=== CHAT INTERACTIVO CON MEMORIA ===")
print("Escribe 'salir' para terminar\n")

print("Algunas preguntas sugeridas para probar:")
print("  1. ¿Cuál es el nombre del jugador?")
print("  2. ¿Dónde nació?")
print("  3. Tradúceme eso al inglés")
print("  4. ¿Qué títulos ganó con Argentina?")
print("  5. ¿Cuántos fueron en total?")

while True:
    user_input = input("👤 Tú: ")

    if user_input.lower() in ['salir', 'exit', 'quit']:
        print("\n¡Hasta luego!")
        break

    if user_input.strip() == "":
        continue

    response = chat_interactive(user_input)
    print(f"🤖 Asistente: {response}\n")

=== CHAT INTERACTIVO CON MEMORIA ===
Escribe 'salir' para terminar

Algunas preguntas sugeridas para probar:
  1. ¿Cuál es el nombre del jugador?
  2. ¿Dónde nació?
  3. Tradúceme eso al inglés
  4. ¿Qué títulos ganó con Argentina?
  5. ¿Cuántos fueron en total?
👤 Tú: Hola
🤖 Asistente: Hola. ¿En qué puedo ayudarte hoy con la información sobre Lionel Messi?

👤 Tú: A partir de ahora, respondeme todo en ingles
🤖 Asistente: Okay, from now on, I will respond in English. How can I assist you?

👤 Tú: Donde nació Messi?
🤖 Asistente: He was born in Rosario, Santa Fe, Argentina.

👤 Tú: salir

¡Hasta luego!


### **Trabajos con múltiples documentos**


In [18]:
# Creamos una base de conocimiento con múltiples documentos
documentos_db = {
    "messi": """Lionel Andrés Messi Cuccittini (Rosario, 24 de junio de 1987), conocido como Leo Messi, es un futbolista argentino que juega como delantero o centrocampista. Desde 2023, integra el plantel del Inter Miami de la MLS canadoestadounidense. Es también internacional con la selección de Argentina, de la que es capitán. Ha ganado 8 Balones de Oro y múltiples títulos incluyendo el Mundial 2022.""",

    "ronaldo": """Cristiano Ronaldo dos Santos Aveiro (Funchal, 5 de febrero de 1985), conocido como Cristiano Ronaldo, es un futbolista portugués que juega como delantero. Ha jugado en clubes como Manchester United, Real Madrid y Juventus. Ha ganado 5 Balones de Oro y es uno de los máximos goleadores de la historia del fútbol. Ganó la Eurocopa 2016 con Portugal.""",

    "maradona": """Diego Armando Maradona (Lanús, 30 de octubre de 1960 - Tigre, 25 de noviembre de 2020) fue un futbolista y entrenador argentino. Es considerado uno de los mejores jugadores de la historia del fútbol. Jugó en Boca Juniors, Barcelona y Napoli entre otros. Ganó el Mundial de México 1986 con Argentina, donde anotó el famoso 'Gol del Siglo' contra Inglaterra.""",

    "pele": """Edson Arantes do Nascimento, conocido como Pelé (Tres Corazones, 23 de octubre de 1940 - São Paulo, 29 de diciembre de 2022), fue un futbolista brasileño. Es considerado por muchos como el mejor jugador de la historia. Ganó tres Copas del Mundo con Brasil (1958, 1962 y 1970). Anotó más de 1000 goles en su carrera profesional."""
}

In [19]:
# Generamos la función

def chat_con_seleccion_documento(pregunta, documento_clave):
    """
    Permite hacer preguntas sobre un documento específico seleccionado por el usuario

    Args:
        pregunta: La pregunta del usuario
        documento_clave: La clave del documento a consultar (ej: "messi", "ronaldo")

    Returns:
        La respuesta basada en el documento seleccionado
    """
    if documento_clave not in documentos_db:
        return f"❌ Error: Documento '{documento_clave}' no encontrado. Documentos disponibles: {', '.join(documentos_db.keys())}"

    documento = documentos_db[documento_clave]

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"Eres un asistente experto. Responde basándote ÚNICAMENTE en este documento:\n\n{documento}"),
        ("user", "{question}")
    ])

    chain = prompt | llm
    response = chain.invoke({"question": pregunta})

    return response.content

# Ejemplos de uso con diferentes documentos
print("=== SISTEMA DE SELECCIÓN DE DOCUMENTOS ===\n")
print(f"📚 Documentos disponibles: {', '.join(documentos_db.keys())}\n")
print("=" * 80 + "\n")

# Consulta sobre Messi
print("📄 Consultando documento: MESSI")
print("👤 Pregunta: ¿Cuántos Balones de Oro ganó?")
resp1 = chat_con_seleccion_documento("¿Cuántos Balones de Oro ganó?", "messi")
print(f"🤖 Respuesta: {resp1}\n")
print("-" * 80 + "\n")

# Consulta sobre Ronaldo
print("📄 Consultando documento: RONALDO")
print("👤 Pregunta: ¿Cuántos Balones de Oro ganó?")
resp2 = chat_con_seleccion_documento("¿Cuántos Balones de Oro ganó?", "ronaldo")
print(f"🤖 Respuesta: {resp2}\n")
print("-" * 80 + "\n")

# Consulta sobre Maradona
print("📄 Consultando documento: MARADONA")
print("👤 Pregunta: ¿Qué mundial ganó?")
resp3 = chat_con_seleccion_documento("¿Qué mundial ganó?", "maradona")
print(f"🤖 Respuesta: {resp3}\n")
print("-" * 80 + "\n")

# Consulta sobre Pelé
print("📄 Consultando documento: PELE")
print("👤 Pregunta: ¿Cuántos mundiales ganó?")
resp4 = chat_con_seleccion_documento("¿Cuántos mundiales ganó?", "pele")
print(f"🤖 Respuesta: {resp4}\n")
print("-" * 80 + "\n")

=== SISTEMA DE SELECCIÓN DE DOCUMENTOS ===

📚 Documentos disponibles: messi, ronaldo, maradona, pele


📄 Consultando documento: MESSI
👤 Pregunta: ¿Cuántos Balones de Oro ganó?
🤖 Respuesta: Ha ganado 8 Balones de Oro.

--------------------------------------------------------------------------------

📄 Consultando documento: RONALDO
👤 Pregunta: ¿Cuántos Balones de Oro ganó?
🤖 Respuesta: Ganó 5 Balones de Oro.

--------------------------------------------------------------------------------

📄 Consultando documento: MARADONA
👤 Pregunta: ¿Qué mundial ganó?
🤖 Respuesta: Ganó el Mundial de México 1986.

--------------------------------------------------------------------------------

📄 Consultando documento: PELE
👤 Pregunta: ¿Cuántos mundiales ganó?


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 40.050023087s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

#### Selección automática de documento relevante

In [ ]:
def seleccionar_documento_automatico(pregunta):
    """
    Sistema que selecciona automáticamente el documento más relevante basándose en la pregunta
    Este es un ejemplo SIMPLE de RAG (Retrieval-Augmented Generation)

    Args:
        pregunta: La pregunta del usuario

    Returns:
        Tupla con (nombre_documento, respuesta)
    """
    # Paso 1: Usar el LLM para determinar sobre qué jugador es la pregunta
    prompt_seleccion = ChatPromptTemplate.from_messages([
        ("system", f"""Eres un clasificador de preguntas.
        Tienes estos documentos disponibles: {', '.join(documentos_db.keys())}

        Tu tarea es determinar a cuál de estos jugadores se refiere la pregunta del usuario.
        Responde ÚNICAMENTE con el nombre del jugador en minúsculas (messi, ronaldo, maradona o pele).
        Si no estás seguro o la pregunta no es sobre ninguno de ellos, responde 'ninguno'."""),
        ("user", "{question}")
    ])

    chain_seleccion = prompt_seleccion | llm
    seleccion = chain_seleccion.invoke({"question": pregunta})
    documento_seleccionado = seleccion.content.strip().lower()

    # Paso 2: Si encontramos el documento, hacer la consulta
    if documento_seleccionado in documentos_db:
        documento = documentos_db[documento_seleccionado]

        prompt_respuesta = ChatPromptTemplate.from_messages([
            ("system", f"Responde basándote en este documento:\n\n{documento}"),
            ("user", "{question}")
        ])

        chain_respuesta = prompt_respuesta | llm
        respuesta = chain_respuesta.invoke({"question": pregunta})

        return documento_seleccionado, respuesta.content
    else:
        return "ninguno", "No pude identificar sobre qué jugador es tu pregunta. Por favor, sé más específico."

# Probamos el sistema de selección automática
print("=== SISTEMA DE SELECCIÓN AUTOMÁTICA DE DOCUMENTOS (RAG BÁSICO) ===\n")
print("El sistema determina automáticamente qué documento usar según tu pregunta\n")
print("=" * 80 + "\n")

# Ejemplo 1
pregunta1 = "¿Cuándo nació el jugador que ganó el mundial 2022?"
print(f"👤 Pregunta: {pregunta1}")
doc1, resp1 = seleccionar_documento_automatico(pregunta1)
print(f"🔍 Documento seleccionado: {doc1.upper()}")
print(f"🤖 Respuesta: {resp1}\n")
print("-" * 80 + "\n")

# Ejemplo 2
pregunta2 = "¿En qué equipos jugó el futbolista portugués?"
print(f"👤 Pregunta: {pregunta2}")
doc2, resp2 = seleccionar_documento_automatico(pregunta2)
print(f"🔍 Documento seleccionado: {doc2.upper()}")
print(f"🤖 Respuesta: {resp2}\n")
print("-" * 80 + "\n")

# Ejemplo 3
pregunta3 = "¿Quién hizo el Gol del Siglo?"
print(f"👤 Pregunta: {pregunta3}")
doc3, resp3 = seleccionar_documento_automatico(pregunta3)
print(f"🔍 Documento seleccionado: {doc3.upper()}")
print(f"🤖 Respuesta: {resp3}\n")
print("-" * 80 + "\n")

# Ejemplo 4
pregunta4 = "¿Cuántos mundiales ganó el Rey del fútbol?"
print(f"👤 Pregunta: {pregunta4}")
doc4, resp4 = seleccionar_documento_automatico(pregunta4)
print(f"🔍 Documento seleccionado: {doc4.upper()}")
print(f"🤖 Respuesta: {resp4}\n")
print("-" * 80 + "\n")

print("💡 Este es un ejemplo básico de RAG (Retrieval-Augmented Generation)")
print("   En sistemas reales se usan embeddings y bases de datos vectoriales")

=== SISTEMA DE SELECCIÓN AUTOMÁTICA DE DOCUMENTOS (RAG BÁSICO) ===

El sistema determina automáticamente qué documento usar según tu pregunta


👤 Pregunta: ¿Cuándo nació el jugador que ganó el mundial 2022?
🔍 Documento seleccionado: MESSI
🤖 Respuesta: El jugador que ganó el Mundial 2022 (Lionel Andrés Messi Cuccittini) nació el **24 de junio de 1987**.

--------------------------------------------------------------------------------

👤 Pregunta: ¿En qué equipos jugó el futbolista portugués?
🔍 Documento seleccionado: RONALDO
🤖 Respuesta: El futbolista portugués jugó en clubes como Manchester United, Real Madrid y Juventus.

--------------------------------------------------------------------------------

👤 Pregunta: ¿Quién hizo el Gol del Siglo?
🔍 Documento seleccionado: MARADONA
🤖 Respuesta: Diego Armando Maradona hizo el Gol del Siglo.

--------------------------------------------------------------------------------

👤 Pregunta: ¿Cuántos mundiales ganó el Rey del fútbol?
🔍 Documento s

### Sistema multi-documento con memoria conversacional

In [ ]:
# Sistema avanzado que combina múltiples documentos CON memoria conversacional
class ChatMultiDocumento:
    def __init__(self, documentos_dict):
        """
        Inicializa el chat con múltiples documentos

        Args:
            documentos_dict: Diccionario con {nombre: contenido} de documentos
        """
        self.documentos = documentos_dict
        self.documento_actual = None
        self.historial = []

    def cambiar_documento(self, nombre_documento):
        """Cambia el documento activo"""
        if nombre_documento in self.documentos:
            self.documento_actual = nombre_documento
            # Reiniciamos el historial con el nuevo documento
            self.historial = [
                SystemMessage(content=f"""Eres un asistente experto. Recuerda toda la conversación.
                Documento actual: {nombre_documento.upper()}

                {self.documentos[nombre_documento]}

                Responde basándote en este documento.""")
            ]
            return f"✅ Documento cambiado a: {nombre_documento.upper()}"
        else:
            return f"❌ Documento '{nombre_documento}' no encontrado"

    def preguntar(self, pregunta):
        """Hace una pregunta sobre el documento actual"""
        if not self.documento_actual:
            return "⚠️ Primero debes seleccionar un documento con cambiar_documento()"

        # Agregamos la pregunta al historial
        self.historial.append(HumanMessage(content=pregunta))

        # Obtenemos respuesta
        response = llm.invoke(self.historial)

        # Agregamos la respuesta al historial
        self.historial.append(AIMessage(content=response.content))

        return response.content

    def listar_documentos(self):
        """Lista los documentos disponibles"""
        return list(self.documentos.keys())

    def obtener_estadisticas(self):
        """Obtiene estadísticas del chat actual"""
        if not self.historial:
            return "No hay historial activo"

        num_mensajes = len(self.historial) - 1  # Restamos el mensaje del sistema
        num_preguntas = num_mensajes // 2

        return f"""
        📊 Estadísticas:
        - Documento actual: {self.documento_actual.upper() if self.documento_actual else 'Ninguno'}
        - Mensajes en historial: {num_mensajes}
        - Preguntas realizadas: {num_preguntas}
        """

# Creamos el sistema
chat_sistema = ChatMultiDocumento(documentos_db)

print("=== SISTEMA MULTI-DOCUMENTO CON MEMORIA ===\n")
print(f"📚 Documentos disponibles: {', '.join(chat_sistema.listar_documentos())}\n")
print("=" * 80 + "\n")

# Conversación sobre Messi
print("📄 " + chat_sistema.cambiar_documento("messi"))
print()

print("👤 Usuario: ¿Cuál es el nombre completo del jugador?")
resp1 = chat_sistema.preguntar("¿Cuál es el nombre completo del jugador?")
print(f"🤖 Asistente: {resp1}\n")
print("-" * 40 + "\n")

print("👤 Usuario: ¿Cuántos Balones de Oro tiene?")
resp2 = chat_sistema.preguntar("¿Cuántos Balones de Oro tiene?")
print(f"🤖 Asistente: {resp2}\n")
print("-" * 40 + "\n")

print("👤 Usuario: ¿Cuál es su apodo?")
resp3 = chat_sistema.preguntar("¿Cuál es su apodo?")
print(f"🤖 Asistente: {resp3}\n")
print("=" * 80 + "\n")

# Cambiamos a otro documento
print("📄 " + chat_sistema.cambiar_documento("maradona"))
print()

print("👤 Usuario: ¿Dónde y cuándo nació este jugador?")
resp4 = chat_sistema.preguntar("¿Dónde y cuándo nació este jugador?")
print(f"🤖 Asistente: {resp4}\n")
print("-" * 40 + "\n")

print("👤 Usuario: ¿Qué gol famoso hizo?")
resp5 = chat_sistema.preguntar("¿Qué gol famoso hizo?")
print(f"🤖 Asistente: {resp5}\n")
print("-" * 40 + "\n")

print("👤 Usuario: Tradúceme eso al inglés")
resp6 = chat_sistema.preguntar("Tradúceme eso al inglés")
print(f"🤖 Asistente: {resp6}\n")
print("=" * 80 + "\n")

# Mostramos estadísticas
print(chat_sistema.obtener_estadisticas())

print("\\n💡 Este sistema combina:")
print("   ✓ Múltiples documentos")
print("   ✓ Memoria conversacional")
print("   ✓ Cambio dinámico de contexto")

=== SISTEMA MULTI-DOCUMENTO CON MEMORIA ===

📚 Documentos disponibles: messi, ronaldo, maradona, pele


📄 ✅ Documento cambiado a: MESSI

👤 Usuario: ¿Cuál es el nombre completo del jugador?
🤖 Asistente: El nombre completo del jugador es Lionel Andrés Messi Cuccittini.

----------------------------------------

👤 Usuario: ¿Cuántos Balones de Oro tiene?
🤖 Asistente: Tiene 8 Balones de Oro.

----------------------------------------

👤 Usuario: ¿Cuál es su apodo?
🤖 Asistente: Su apodo es Leo Messi.


📄 ✅ Documento cambiado a: MARADONA

👤 Usuario: ¿Dónde y cuándo nació este jugador?
🤖 Asistente: Este jugador nació en Lanús el 30 de octubre de 1960.

----------------------------------------

👤 Usuario: ¿Qué gol famoso hizo?
🤖 Asistente: Hizo el famoso 'Gol del Siglo' contra Inglaterra.

----------------------------------------

👤 Usuario: Tradúceme eso al inglés
🤖 Asistente: He scored the famous 'Goal of the Century' against England.



        📊 Estadísticas:
        - Documento actual: MARA

### Transformación de query en lenguaje natural a SQL

In [ ]:
# Verificamos afirmaciones contra el documento
prompt_factcheck = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de transformación de queries en lenguaje natural a {system}, asegurate de solamente devolver el fragmento de código y nada más"""),
    ("user", "Necesito que me conviertas esto: {query}")
])

chain_factcheck = prompt_factcheck | llm

# Afirmaciones para verificar
queries = [
    "¿Cuantos clientes forman parte de nuestra Base de Datos?",
    "Quien fue el vendedor que más vendió?",
    "Cuál fue la película más taquillera del año 2021?",
    "Cuál fue la película más taquillera del año 2021? Y que además ganó al menos 1 oscar entre película y reparto"
]

print("=== CONVERSIÓN DE CONSULTAS ===\n")
for query in queries:
    response = chain_factcheck.invoke({
        "system": "SQL",
        "query": query
    })
    print(f"Afirmación: {query}")
    print(f"Verificación: {response.content}\n")
    print("-" * 80 + "\n")

=== CONVERSIÓN DE CONSULTAS ===

Afirmación: ¿Cuantos clientes forman parte de nuestra Base de Datos?
Verificación: ```sql
SELECT COUNT(*) FROM customers;
```

--------------------------------------------------------------------------------

Afirmación: Quien fue el vendedor que más vendió?
Verificación: ```sql
SELECT
  s.Name
FROM Sales AS sl
JOIN Salespeople AS s
  ON sl.SalespersonID = s.SalespersonID
GROUP BY
  s.SalespersonID,
  s.Name
ORDER BY
  SUM(sl.Amount) DESC
LIMIT 1;
```

--------------------------------------------------------------------------------

Afirmación: Cuál fue la película más taquillera del año 2021?
Verificación: ```sql
SELECT title
FROM movies
WHERE release_year = 2021
ORDER BY gross_revenue DESC
LIMIT 1;
```

--------------------------------------------------------------------------------

Afirmación: Cuál fue la película más taquillera del año 2021? Y que además ganó al menos 1 oscar entre película y reparto
Verificación: ```sql
SELECT
    P.titulo,
    P.

#### Transformación con más información

In [ ]:
# Verificamos afirmaciones contra el documento
prompt_factcheck = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de transformación de queries en lenguaje natural a {system}, asegurate de solamente devolver el fragmento de código y nada más.
    La información de la base de datos es la siguiente: {informacion}"""),
    ("user", "Necesito que me conviertas esto: {query}")
])

chain_factcheck = prompt_factcheck | llm

# Afirmaciones para verificar
queries = [
    "¿Cuantos clientes forman parte de nuestra Base de Datos?",
    "Quien fue el vendedor que más vendió?",
    "Cuál fue la película más taquillera del año 2021?",
    "Cuál fue la película más taquillera del año 2021? Y que además ganó al menos 1 oscar entre película y reparto"
]

informacion = """CREATE TABLE zt_clientes (
    id_cliente INT PRIMARY KEY,
    nombre VARCHAR(100),
    pais VARCHAR(50)
);

CREATE TABLE zt_vendedores (
    id_vendedor INT PRIMARY KEY,
    nombre VARCHAR(100),
    region VARCHAR(50)
);

CREATE TABLE zt_peliculas (
    id_pelicula INT PRIMARY KEY,
    titulo VARCHAR(150),
    anio INT,
    taquilla_total DECIMAL(15,2),
    gano_oscar BOOLEAN
);

CREATE TABLE zt_ventas (
    id_venta INT PRIMARY KEY,
    id_vendedor INT,
    id_cliente INT,
    monto DECIMAL(10,2),
    FOREIGN KEY (id_vendedor) REFERENCES zt_vendedores(id_vendedor),
    FOREIGN KEY (id_cliente) REFERENCES zt_clientes(id_cliente)
);"""


print("=== CONVERSIÓN DE CONSULTAS ===\n")
for query in queries:
    response = chain_factcheck.invoke({
        "system": "SQL",
        "informacion": informacion,
        "query": query
    })
    print(f"Afirmación: {query}")
    print(f"Verificación: {response.content}\n")
    print("-" * 80 + "\n")

=== CONVERSIÓN DE CONSULTAS ===

Afirmación: ¿Cuantos clientes forman parte de nuestra Base de Datos?
Verificación: ```sql
SELECT COUNT(id_cliente) FROM zt_clientes;
```

--------------------------------------------------------------------------------

Afirmación: Quien fue el vendedor que más vendió?
Verificación: ```sql
SELECT
  v.nombre
FROM zt_vendedores AS v
JOIN zt_ventas AS ve
  ON v.id_vendedor = ve.id_vendedor
GROUP BY
  v.nombre
ORDER BY
  SUM(ve.monto) DESC
LIMIT 1;
```

--------------------------------------------------------------------------------

Afirmación: Cuál fue la película más taquillera del año 2021?
Verificación: ```sql
SELECT titulo
FROM zt_peliculas
WHERE anio = 2021
ORDER BY taquilla_total DESC
LIMIT 1;
```

--------------------------------------------------------------------------------

Afirmación: Cuál fue la película más taquillera del año 2021? Y que además ganó al menos 1 oscar entre película y reparto
Verificación: ```sql
SELECT titulo
FROM zt_pelicula

## Ejercicios prácticos a resolver

### 📝 Ejercicio 1: Base de datos de películas

**Objetivo:** Crear un sistema de consulta sobre películas

**Instrucciones:**
1. Crea un diccionario con información de al menos 3 películas diferentes (título, director, año, sinopsis, actores principales)
2. Implementa una función que permita hacer preguntas sobre una película específica
3. Prueba con preguntas como:
   - "¿Quién dirigió esta película?"
   - "¿Cuál es la sinopsis?"
   - "¿Qué actores participaron?"
4. Además, crea una función que extraiga los datos de la película en formato JSON.

**Código inicial:**

In [ ]:
# EJERCICIO 1: Completa el código

peliculas_db = {
    "inception": """Inception (2010) es una película dirigida por Christopher Nolan.
    La trama sigue a Dom Cobb, interpretado por Leonardo DiCaprio, un ladrón especializado
    en extraer secretos del subconsciente durante el sueño. Otros actores principales incluyen
    a Marion Cotillard, Tom Hardy y Ellen Page.""",

    "avatar": """Ambientada en el año 2154, Avatar es una épica película de ciencia ficción dirigida por James Cameron que combina aventura, fantasía y tecnología revolucionaria. La historia transcurre en Pandora, una exuberante luna del sistema Alfa Centauri habitada por los Na’vi, una raza humanoide azul que vive en total armonía con la naturaleza y el espíritu de su planeta. La humanidad, habiendo agotado sus recursos en la Tierra, llega a Pandora para extraer un valioso mineral llamado unobtainium. Para interactuar con los nativos, los humanos utilizan “avatares”, cuerpos biológicamente creados que combinan ADN humano y Na’vi y que son controlados a distancia. Jake Sully, un exmarine parapléjico interpretado por Sam Worthington, es reclutado para el programa y, bajo la guía de la Dra. Grace Augustine (Sigourney Weaver) y la piloto Trudy Chacón (Michelle Rodriguez), se infiltra en la comunidad Na’vi. Allí conoce a Neytiri (Zoe Saldaña), quien lo introduce a las costumbres, creencias y la conexión espiritual de su pueblo con Eywa, la fuerza vital del planeta. A medida que Jake aprende su idioma —creado especialmente por el lingüista Paul Frommer— y se enamora de Neytiri, se ve dividido entre su deber militar bajo el mando del coronel Miles Quaritch (Stephen Lang) y su creciente lealtad hacia los Na’vi. La película, estrenada en 2009, costó alrededor de 237 millones de dólares y recaudó más de 2.9 mil millones, convirtiéndose en la película más taquillera de la historia hasta ser momentáneamente superada por Avengers: Endgame en 2019. Filmada en Nueva Zelanda y Los Ángeles con tecnología pionera en captura de movimiento y cámaras 3D desarrolladas por Cameron, Avatar marcó un antes y un después en los efectos visuales y ganó tres premios Óscar por dirección artística, fotografía y efectos visuales. Su mensaje ecologista, su crítica al colonialismo y su universo inmersivo convirtieron a Pandora en un símbolo de conexión con la naturaleza y respeto cultural, dando origen a secuelas como Avatar: The Way of Water (2022) y otras planeadas para la próxima década.""",

    "avengers": """Endgame es una película de superhéroes estadounidense de 2019 dirigida por Anthony y Joe Russo y producida por Marvel Studios, que culmina la saga del Infinito iniciada más de una década antes en el Universo Cinematográfico de Marvel (UCM). Ambientada después de los devastadores eventos de Avengers: Infinity War (2018), la historia comienza con un universo sumido en la desesperación tras el chasquido de Thanos (Josh Brolin), que eliminó a la mitad de todos los seres vivos. Los héroes sobrevivientes —Tony Stark/Iron Man (Robert Downey Jr.), Steve Rogers/Capitán América (Chris Evans), Thor (Chris Hemsworth), Natasha Romanoff/Black Widow (Scarlett Johansson), Bruce Banner/Hulk (Mark Ruffalo), y Clint Barton/Hawkeye (Jeremy Renner)— intentan recomponerse mientras enfrentan la pérdida y la culpa. Cuando Scott Lang/Ant-Man (Paul Rudd) emerge del Reino Cuántico con la idea de viajar en el tiempo, los Vengadores conciben un arriesgado plan para recuperar las Gemas del Infinito en distintas líneas temporales y revertir el chasquido. Esto los lleva a revivir momentos icónicos de películas anteriores del UCM, enfrentarse a versiones pasadas de sus enemigos y aceptar su destino. Con la ayuda de Rocket, Nebula, Okoye, Capitán Marvel (Brie Larson) y otros héroes, se desencadena una batalla final monumental contra Thanos. La película combina acción, emoción y cierre narrativo para más de veinte películas anteriores. Con un presupuesto estimado de 356 millones de dólares, Endgame recaudó más de 2.799 millones, convirtiéndose en la película más taquillera de la historia hasta el reestreno de Avatar en 2021. Fue aclamada por su dirección, sus efectos visuales, la actuación de su elenco coral y su capacidad para ofrecer una conclusión épica a una saga de más de diez años. Ganó múltiples premios, fue nominada al Óscar a mejores efectos visuales y es considerada uno de los mayores logros del cine contemporáneo de superhéroes."""
}

def consultar_pelicula(pregunta, nombre_pelicula):
    # TODO: Implementa la función que responda preguntas sobre la película
    # Pista: Usa ChatPromptTemplate y el patrón que vimos en los ejemplos
    pass

# TODO: Prueba tu función con diferentes preguntas
# print(consultar_pelicula("¿Quién dirigió esta película?", "inception"))

### 📝 Ejercicio 2: Chat con memoria sobre países

**Objetivo:** Crear un chat conversacional con memoria que responda preguntas sobre países

**Instrucciones:**
1. Crea un documento con información detallada sobre un país (capital, población, idioma, moneda, atracciones turísticas)
2. Implementa un sistema de chat con memoria conversacional
3. Realiza una conversación de al menos 5 preguntas donde cada pregunta haga referencia a la anterior

**Ejemplo de conversación:**
- "¿Cuál es la capital?"
- "¿Cuántos habitantes tiene esa ciudad?"
- "¿Me lo puedes decir en millones?"
- "¿Qué idioma se habla allí?"
- "Dame un ejemplo de frase en ese idioma"

**Código inicial:**

In [ ]:
# EJERCICIO 2: Completa el código

# TODO: Crea un documento con información sobre un país
documento_pais = """
# Escribe acá información detallada sobre un país de tu elección
"""

# TODO: Inicializa el historial de mensajes con el documento como contexto
historial_pais = [
    # Agrega acá el SystemMessage con el documento
]

def chat_pais(pregunta):
    # TODO: Implementa la función que mantenga memoria conversacional
    # Pista: Usa HumanMessage y AIMessage como vimos en el Ejemplo 2
    pass

# TODO: Realiza una conversación de 5 preguntas que hagan referencia entre sí
# Ejemplo:
# print("P:", "¿Cuál es la capital?")
# print("R:", chat_pais("¿Cuál es la capital?"))
# print()

### 📝 Ejercicio 3: Selector automático de documentos (RAG)

**Objetivo:** Crear un sistema que seleccione automáticamente el documento correcto según la pregunta

**Instrucciones:**
1. Crea una base de datos con información de al menos 3 lenguajes de programación
2. Implementa un sistema que identifique automáticamente sobre qué lenguaje pregunta el usuario
3. Responde la pregunta usando el documento correcto

**Ejemplo:**
- Pregunta: "¿Qué tipo de lenguaje es el que fue creado por Guido van Rossum?"
- Sistema selecciona: "python"
- Respuesta basada en el documento de Python

**Código inicial:**

In [ ]:
# EJERCICIO 3: Completa el código

lenguajes_db = {
    "python": """Python es un lenguaje interpretado de alto nivel creado por Guido van Rossum.
    Se caracteriza por su sintaxis clara y es muy usado en data science y machine learning.""",

    # TODO: Agrega información de 2 lenguajes más (Java, JavaScript, C++, etc.)
    # "java": "...",
    # "javascript": "...",
}

def seleccionar_y_responder(pregunta):
    # TODO: Paso 1 - Usa el LLM para identificar sobre qué lenguaje es la pregunta
    # TODO: Paso 2 - Obtén el documento correspondiente
    # TODO: Paso 3 - Responde la pregunta usando ese documento
    # TODO: Retorna una tupla (lenguaje_seleccionado, respuesta)
    pass

# TODO: Prueba con preguntas que requieran selección automática
# preguntas_prueba = [
#     "¿Qué lenguaje usa principalmente en data science?",
#     "¿Cuál lenguaje se ejecuta en el navegador?",
#     "¿Qué lenguaje es orientado a objetos y compilado?"
# ]

# for pregunta in preguntas_prueba:
#     lenguaje, respuesta = seleccionar_y_responder(pregunta)
#     print(f"Pregunta: {pregunta}")
#     print(f"Lenguaje detectado: {lenguaje}")
#     print(f"Respuesta: {respuesta}")
#     print()

### 📝 Ejercicio 4: Sistema de lectura de archivos con comparación

**Objetivo:** Leer múltiples archivos y compararlos

**Instrucciones:**
1. Crea 2-3 archivos .txt con información sobre diferentes tecnologías
2. Implementa una función que lea los archivos
3. Crea una función que compare la información de los archivos
4. Haz preguntas comparativas como "¿Cuál es más antiguo?" o "¿Cuál es más rápido?"

**Código inicial:**

In [ ]:
# EJERCICIO 4: Completa el código

# TODO: Paso 1 - Crea archivos con información
# Ejemplo:
# archivo1 = "react_info.txt" con información sobre React
# archivo2 = "vue_info.txt" con información sobre Vue
# archivo3 = "angular_info.txt" con información sobre Angular

def crear_archivos_tecnologias():
    # TODO: Crea los archivos con información relevante
    tecnologias = {
        "react_info.txt": "Escribe información sobre React aquí...",
        "vue_info.txt": "Escribe información sobre Vue aquí...",
        # Agrega más archivos
    }

    # for nombre_archivo, contenido in tecnologias.items():
    #     with open(nombre_archivo, 'w', encoding='utf-8') as f:
    #         f.write(contenido)
    pass

def comparar_tecnologias(pregunta_comparativa, archivos):
    # TODO: Lee los archivos y combina su contenido
    # TODO: Usa un prompt que pida al LLM comparar la información
    # TODO: Retorna la comparación
    pass

# TODO: Prueba tu sistema
# crear_archivos_tecnologias()
# comparacion = comparar_tecnologias(
#     "¿Cuál framework es más fácil de aprender?",
#     ["react_info.txt", "vue_info.txt", "angular_info.txt"]
# )
# print(comparacion)

### 📝 Ejercicio 5: Sistema completo - Asistente educativo

**Objetivo:** Crear un asistente educativo completo que combine todo lo aprendido

**Instrucciones:**
1. Crea una base de datos con al menos 5 temas educativos diferentes (historia, ciencias, matemáticas, etc.)
2. Implementa selección automática de tema
3. Agrega memoria conversacional
4. Permite cambiar de tema manteniendo conversaciones separadas
5. Incluye estadísticas y resúmenes

**Características que debe tener:**
- ✅ Múltiples documentos organizados por tema
- ✅ Selección automática del tema según la pregunta
- ✅ Memoria conversacional por tema
- ✅ Capacidad de cambiar de tema
- ✅ Generación de resúmenes de la conversación
- ✅ Estadísticas de uso

**Código inicial:**

In [ ]:
# EJERCICIO 5: Proyecto final - Sistema completo

class AsistenteEducativo:
    def __init__(self):
        # TODO: Define la base de conocimiento con múltiples temas
        self.temas = {
            "matematicas": "Información sobre matemáticas...",
            "historia": "Información sobre historia...",
            # Agrega más temas
        }

        # TODO: Diccionario para guardar historiales por tema
        self.historiales = {}
        self.tema_actual = None

    def seleccionar_tema_automatico(self, pregunta):
        # TODO: Identifica el tema de la pregunta
        pass

    def cambiar_tema(self, nombre_tema):
        # TODO: Cambia el tema actual
        # TODO: Inicializa o recupera el historial de ese tema
        pass

    def preguntar(self, pregunta):
        # TODO: Responde manteniendo memoria conversacional
        pass

    def generar_resumen_conversacion(self):
        # TODO: Genera un resumen de la conversación actual
        pass

    def obtener_estadisticas(self):
        # TODO: Retorna estadísticas de uso (temas consultados, preguntas por tema, etc.)
        pass

# TODO: Prueba tu asistente
# asistente = AsistenteEducativo()
#
# # Ejemplo de uso:
# asistente.cambiar_tema("matematicas")
# print(asistente.preguntar("¿Qué es un número primo?"))
# print(asistente.preguntar("Dame un ejemplo"))
#
# asistente.cambiar_tema("historia")
# print(asistente.preguntar("¿Qué pasó en 1492?"))
#
# print(asistente.obtener_estadisticas())
# print(asistente.generar_resumen_conversacion())